# FMVA v0.5 — preflight and prospective protocol freeze

Run once on a CPU runtime before either GPU notebook. It freezes the new 4 August 2026
information cutoff, checks all real-cell matrices and checkpoints, records immutable hashes, and
writes no future labels or model scores.

In [ ]:
# ruff: noqa: E402
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# @title 1. Install the frozen FMVA v0.5 alpha runtime
import subprocess
import sys
from pathlib import Path

DRIVE_NOTEBOOK_DIRS = [
    Path(
        "/content/drive/MyDrive/projects_bioinfo_2026/fm-value-audit-real/temporal_benchmark/08_notebooks"
    ),
    Path("/content/drive/MyDrive/fm-value-audit-real/temporal_benchmark/08_notebooks"),
]
wheel_candidates = []
for directory in DRIVE_NOTEBOOK_DIRS:
    if directory.is_dir():
        wheel_candidates.extend(directory.glob("fm_value_audit-0.5.0a1-py3-none-any.whl"))
if len(wheel_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one fm_value_audit-0.5.0a1 wheel in temporal_benchmark/08_notebooks; "
        f"found {wheel_candidates}"
    )
WHEEL = wheel_candidates[0]
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "anndata>=0.11.4,<0.12",
        "h5py>=3.11,<4",
        "safetensors>=0.4",
        "pyarrow>=16",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(WHEEL)], check=True)
print("Installed:", WHEEL)

In [ ]:
# ruff: noqa: E402
# @title 2. Mount Drive and locate the canonical project
from google.colab import drive

drive.mount("/content/drive")

import hashlib
import json
import platform
import time
from datetime import UTC, datetime
from pathlib import Path

import anndata as ad
import pandas as pd
import torch

PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/projects_bioinfo_2026/fm-value-audit-real"),
    Path("/content/drive/MyDrive/fm-value-audit-real"),
]
PROJECT = next((path for path in PROJECT_CANDIDATES if path.is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError(f"Could not locate fm-value-audit-real; checked {PROJECT_CANDIDATES}")

TEMPORAL_V05 = PROJECT / "temporal_benchmark_v0.5"
PROTOCOL_DIR = TEMPORAL_V05 / "00_protocol"
CONTEXTUAL_DIR = TEMPORAL_V05 / "04_contextual_embeddings"
PROTOCOL_PATH = PROTOCOL_DIR / "FMVA_CONTEXTUAL_PROSPECTIVE_v0.5.json"
PREFLIGHT_PATH = PROTOCOL_DIR / "PREFLIGHT_MANIFEST.json"

STAGE1_CANDIDATES = [
    PROJECT / "outputs" / "stage1_full_cells_16gb",
    PROJECT / "outputs" / "stage1_full_cells",
]
STAGE1 = next((path for path in STAGE1_CANDIDATES if path.is_dir()), None)
if STAGE1 is None:
    raise FileNotFoundError(
        f"Could not locate full-cell Stage 1 outputs; checked {STAGE1_CANDIDATES}"
    )

MODEL_ROOT = PROJECT / "models"
PATHS = {
    "gse115978_h5ad": STAGE1 / "GSE115978_ALL_7186_raw_counts.h5ad",
    "gse179994_h5ad": STAGE1 / "GSE179994_ALL_150849_raw_counts.h5ad",
    "gse179994_lesion_mapping": STAGE1 / "GSE179994_official_lesion_sample_mapping.csv",
    "geneformer_model": MODEL_ROOT / "Geneformer-V1-10M" / "model.safetensors",
    "geneformer_symbol_map": MODEL_ROOT / "gene_dictionaries_30m" / "gene_name_id_dict_gc30M.pkl",
    "geneformer_tokens": MODEL_ROOT / "gene_dictionaries_30m" / "token_dictionary_gc30M.pkl",
    "geneformer_medians": MODEL_ROOT / "gene_dictionaries_30m" / "gene_median_dictionary_gc30M.pkl",
    "scgpt_model": MODEL_ROOT / "scGPT-whole-human" / "best_model.pt",
    "scgpt_vocab": MODEL_ROOT / "scGPT-whole-human" / "vocab.json",
    "scgpt_args": MODEL_ROOT / "scGPT-whole-human" / "args.json",
}
missing = [str(path) for path in PATHS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

print("Project:", PROJECT)
print("Stage 1:", STAGE1)
print(
    "CUDA:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
)

In [ ]:
# @title 3. Freeze and validate the protocol
from fmva.contextual import default_contextual_protocol, sha256_file, write_default_protocol

PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
if PROTOCOL_PATH.exists():
    existing = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    expected = default_contextual_protocol().model_dump(mode="json")
    if existing != expected:
        raise RuntimeError("Existing v0.5 protocol differs from the packaged frozen protocol")
else:
    write_default_protocol(PROTOCOL_PATH)

protocol = default_contextual_protocol()
print("Protocol:", protocol.protocol_id)
print("Protocol SHA-256:", sha256_file(PROTOCOL_PATH))
print("Primary contrast:", protocol.primary_contrast)
print("First eligible future snapshot:", protocol.snapshot_schedule.first_eligible_snapshot)

In [ ]:
# @title 4. Verify matrices, metadata alignment and checkpoint hashes
EXPECTED_CHECKPOINT_HASHES = {
    "geneformer_model": "a5e33a757431643b3697de7ef6127950cdc49e06e58d4266b3a3ab191b683f14",
    "scgpt_model": "6cb5d451ab5c4b33eb673adbe4fddc61d2389df1b89b7651a9fe2e557572b922",
}


def hash_file(path: Path, chunk_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


file_hashes = {}
for key, path in PATHS.items():
    started = time.perf_counter()
    digest = hash_file(path)
    file_hashes[key] = digest
    print(key, path.stat().st_size, digest, f"{time.perf_counter() - started:.1f}s")

for key, expected in EXPECTED_CHECKPOINT_HASHES.items():
    if file_hashes[key] != expected:
        raise RuntimeError(f"Checkpoint hash mismatch for {key}: {file_hashes[key]} != {expected}")

matrix_summary = {}
for key in ["gse115978_h5ad", "gse179994_h5ad"]:
    adata = ad.read_h5ad(PATHS[key], backed="r")
    try:
        if adata.n_obs == 0 or adata.n_vars == 0:
            raise RuntimeError(f"Empty H5AD: {PATHS[key]}")
        if not adata.obs_names.is_unique or not adata.var_names.is_unique:
            raise RuntimeError(f"Non-unique obs/var identifiers in {PATHS[key]}")
        matrix_summary[key] = {
            "cells": int(adata.n_obs),
            "genes": int(adata.n_vars),
            "obs_columns": [str(value) for value in adata.obs.columns],
            "x_type": type(adata.X).__name__,
        }
        print(key, matrix_summary[key])
    finally:
        adata.file.close()

lesion_mapping = pd.read_csv(PATHS["gse179994_lesion_mapping"])
if lesion_mapping["dataset_sample"].duplicated().any():
    raise RuntimeError("Official lesion mapping contains duplicate dataset samples")
if set(lesion_mapping["lesion_response"]) != {"Responder", "Non-responder"}:
    raise RuntimeError("Unexpected lesion-response values")
print("Official lesion mapping rows:", len(lesion_mapping))

In [ ]:
# @title 5. Write the immutable preflight manifest
from importlib.metadata import version

manifest = {
    "schema_version": "fmva-contextual-preflight-v1",
    "status": "COMPLETE_PROTOCOL_FROZEN_INPUTS_VERIFIED",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "protocol_id": protocol.protocol_id,
    "protocol_sha256": sha256_file(PROTOCOL_PATH),
    "information_cutoff": protocol.information_cutoff,
    "future_labels_accessed": False,
    "model_scores_created": False,
    "paths": {key: str(path) for key, path in PATHS.items()},
    "file_hashes": file_hashes,
    "matrix_summary": matrix_summary,
    "official_lesion_mapping_rows": len(lesion_mapping),
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "anndata": version("anndata"),
        "numpy": version("numpy"),
        "pandas": version("pandas"),
        "torch": version("torch"),
    },
}
PREFLIGHT_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(PROTOCOL_DIR / "V05_PREFLIGHT_COMPLETE.txt").write_text(
    f"v0.5 preflight complete at {manifest['created_at_utc']}\n",
    encoding="utf-8",
)
print(json.dumps(manifest, indent=2)[:5000])